# 04 — NVS Internal Trade Data EDA (Stage 4)

**Run locally. No data leaves your machine.**

### Data properties
- BOM tree structure — each row is a material node in a product bill-of-materials
- ~90 columns; ISO-2 country codes for origin/destination
- 1 timestamp column with 6 distinct snapshot values — format: `YY-MM-DD HH:MM:SS.fffffffff` (nanosecond precision)
- All 6 snapshot timestamps fall within 2025–2026
- Trade values in EUR
- 3 industry classification columns: 2-digit code-text, Industry Group (number), Industry Group text

### How to use
1. Set `NVS_FILE_PATH` in **Cell 1**
2. Fill in your column names in the **COLUMN CONFIG** block in **Cell 1**
3. Run all cells top-to-bottom

---

In [ ]:
# ── Cell 0: Imports ───────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import re

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

OUTPUT_DIR = Path('outputs/nvs_eda')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

C_BLUE  = '#2c7bb6'
C_RED   = '#d7191c'
C_GREEN = '#1a9641'
C_AMBER = '#fdae61'

print('✓ Imports loaded')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Cell 1: FILE PATH + COLUMN CONFIG  ← FILL THIS IN BEFORE RUNNING
# ══════════════════════════════════════════════════════════════════════════════

NVS_FILE_PATH = 'path/to/your/nvs_data.csv'   # <-- UPDATE

# ── Stage 3 corridors (locked) ────────────────────────────────────────────────
STAGE3_CORRIDORS = {
    frozenset(['USA','CHN']), frozenset(['USA','DEU']), frozenset(['USA','IND']),
    frozenset(['USA','BRA']), frozenset(['USA','CAN']), frozenset(['USA','MEX']),
    frozenset(['CHN','DEU']), frozenset(['CHN','IND']), frozenset(['CHN','JPN']),
    frozenset(['CHN','KOR']), frozenset(['DEU','IND']), frozenset(['DEU','BRA']),
    frozenset(['IND','BRA']),
}
STAGE3_KEYS = {'_'.join(sorted(list(p))) for p in STAGE3_CORRIDORS}

ISO2_TO_ISO3 = {
    'US':'USA','CN':'CHN','DE':'DEU','IN':'IND','BR':'BRA',
    'CA':'CAN','MX':'MEX','JP':'JPN','KR':'KOR','FR':'FRA',
    'DK':'DNK','GB':'GBR','AU':'AUS','ZA':'ZAF','RU':'RUS',
    'IT':'ITA','ES':'ESP','NL':'NLD','SE':'SWE','NO':'NOR',
    'PL':'POL','CZ':'CZE','AR':'ARG','CL':'CHL','CO':'COL',
    'EG':'EGY','NG':'NGA','KE':'KEN','TH':'THA','ID':'IDN',
    'MY':'MYS','PH':'PHL','VN':'VNM','PK':'PAK','BD':'BGD',
    'TR':'TUR','SA':'SAU','AE':'ARE','IL':'ISR','SG':'SGP',
}

def iso2_to_iso3(series):
    return series.astype(str).str.strip().str.upper().map(
        lambda x: ISO2_TO_ISO3.get(x, x)
    )

# ──────────────────────────────────────────────────────────────────────────────
#  COLUMN CONFIG — replace placeholders with your actual column names
# ──────────────────────────────────────────────────────────────────────────────

# TIMESTAMP (1 column, 6 distinct snapshot values)
# Values look like: 25-11-13 07:56:08.652466600  (YY-MM-DD HH:MM:SS.nanoseconds)
TIMESTAMP_COL = 'timestamp_col'   # <-- single column name

# COUNTRY LANES  (ISO-2 codes e.g. 'US', 'DE', 'CN')
ORIGIN_COL = 'origin_col'
DEST_COL   = 'destination_col'

# TRADE VALUE (EUR, expected non-negative)
VALUE_COLS = [
    'value_col_1',
]

# BOM STRUCTURE
# BOM_LEVEL_COL  : integer depth — 0 = finished good, 1/2/3 = sub-components
# BOM_PARENT_COL : parent material ID, null for root nodes (or set None)
# BOM_MATERIAL_COL: unique material/item identifier per row
BOM_LEVEL_COL    = 'bom_level_col'
BOM_PARENT_COL   = None
BOM_MATERIAL_COL = 'material_id_col'

# INDUSTRY (3 columns)
# IND_CODE_TEXT_COL : '10 - Food manufacturing'  (2-digit NACE/SIC code + label)
# IND_GROUP_NUM_COL : integer group code e.g. 3, 7, 12
# IND_GROUP_TEXT_COL: 'Animal Health', 'Dairy', etc.
IND_CODE_TEXT_COL  = 'industry_2digit_code_text_col'
IND_GROUP_NUM_COL  = 'industry_group_num_col'
IND_GROUP_TEXT_COL = 'industry_group_text_col'

# BUSINESS FUNCTION (e.g. 'BioAg', 'Human Health') — set None if not present
BF_COL = None

# PRODUCT / MATERIAL DESCRIPTION (free text) — set None if not present
PRODUCT_DESC_COL = None

# ──────────────────────────────────────────────────────────────────────────────
print('✓ Config ready')

---
## Section 1 — Load & Schema Audit

In [ ]:
# ── Cell 2: Load Data ─────────────────────────────────────────────────────────
df = pd.read_csv(NVS_FILE_PATH, low_memory=False)
print(f'✓ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'  Memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
df.head(3)

In [ ]:
# ── Cell 3: Schema Audit ──────────────────────────────────────────────────────
schema = pd.DataFrame({
    'dtype':    df.dtypes.astype(str),
    'nulls':    df.isnull().sum(),
    'null_%':   (df.isnull().mean() * 100).round(1),
    'n_unique': df.nunique(),
    'sample':   [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
})

def highlight_nulls(val):
    if isinstance(val, float):
        if val >= 50: return 'background-color: #f4a582'
        if val >= 10: return 'background-color: #fddbc7'
    return ''

print(f'Full schema — {len(schema)} columns (red = >50% null, orange = >10% null):')
display(schema.style.applymap(highlight_nulls, subset=['null_%']))

In [ ]:
# ── Cell 4: dtype Summary ─────────────────────────────────────────────────────
dtype_counts = df.dtypes.astype(str).value_counts()
for dt, n in dtype_counts.items():
    print(f'  {dt:<15} {n} columns')

fig, ax = plt.subplots(figsize=(6, 3))
dtype_counts.plot(kind='barh', ax=ax, color=C_BLUE, edgecolor='white')
ax.set_title('Column dtype counts')
ax.set_xlabel('Number of columns')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'dtype_summary.png', dpi=120)
plt.show()

---
## Section 2 — Timestamp Analysis
1 timestamp column containing 6 distinct snapshot values, format `YY-MM-DD HH:MM:SS.fffffffff` (nanosecond precision). All within 2025–2026.

In [ ]:
# ── Cell 5: Timestamp Column ─────────────────────────────────────────────────
if TIMESTAMP_COL not in df.columns:
    print(f'⚠  [{TIMESTAMP_COL}] not found — update TIMESTAMP_COL in Cell 1')
    df['_ts_parsed'] = pd.NaT   # placeholder so downstream cells don't crash
else:
    df['_ts_parsed'] = pd.to_datetime(df[TIMESTAMP_COL], errors='coerce')
    ts = df['_ts_parsed']
    print(f'[{TIMESTAMP_COL}]  parsed {ts.notna().sum():,}/{len(df):,}  |  failed {ts.isna().sum():,}')
    print()
    unique_snaps = sorted(ts.dropna().unique())
    print(f'{len(unique_snaps)} distinct snapshot values:')
    for i, s in enumerate(unique_snaps, 1):
        cnt = (ts == s).sum()
        print(f'  [{i}] {s}  →  {cnt:,} rows')

In [ ]:
# ── Cell 6: Row Count per Snapshot ───────────────────────────────────────────
ts = df['_ts_parsed']
if ts.notna().sum() == 0:
    print('⚠  No timestamps parsed — check TIMESTAMP_COL in Cell 1.')
else:
    snap_counts = ts.value_counts().sort_index()
    labels = [str(pd.Timestamp(t))[:19] for t in snap_counts.index]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].bar(range(len(snap_counts)), snap_counts.values, color=C_BLUE, edgecolor='white')
    axes[0].set_xticks(range(len(snap_counts)))
    axes[0].set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    axes[0].set_title('Row count per snapshot')
    axes[0].set_ylabel('Row count')

    axes[1].pie(snap_counts.values, labels=labels, autopct='%1.0f%%', startangle=90,
                colors=plt.cm.Blues(np.linspace(0.3, 0.9, len(snap_counts))))
    axes[1].set_title('Row share per snapshot')

    plt.suptitle(f'Snapshot distribution — [{TIMESTAMP_COL}]', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'snapshot_distribution.png', dpi=120)
    plt.show()

    if VALUE_COLS and VALUE_COLS[0] in df.columns:
        pv = VALUE_COLS[0]
        val_by_snap = (
            df.assign(_val=pd.to_numeric(df[pv], errors='coerce'))
              .groupby('_ts_parsed')['_val'].sum().sort_index()
        )
        print(f'EUR [{pv}] per snapshot:')
        for t, v in val_by_snap.items():
            print(f'  {str(pd.Timestamp(t))[:19]}  →  {v:>15,.0f} EUR')

In [ ]:
# ── Cell 7: EUR Value Trend Across Snapshots ────────────────────────────────
ts = df['_ts_parsed']
if ts.notna().sum() == 0 or not VALUE_COLS or VALUE_COLS[0] not in df.columns:
    print('⚠  Check TIMESTAMP_COL and VALUE_COLS in Cell 1.')
else:
    pv = VALUE_COLS[0]
    snap_val = (
        df.assign(_val=pd.to_numeric(df[pv], errors='coerce'))
          .groupby('_ts_parsed')['_val']
          .agg(total_EUR='sum', avg_EUR='mean', n_rows='count')
          .sort_index()
    )
    snap_val.index = [str(pd.Timestamp(t))[:19] for t in snap_val.index]
    snap_val['delta_EUR'] = snap_val['total_EUR'].diff()
    snap_val['delta_%']   = snap_val['total_EUR'].pct_change() * 100
    display(snap_val.style.format({
        'total_EUR':'{:,.0f}','avg_EUR':'{:,.0f}',
        'delta_EUR':'{:+,.0f}','delta_%':'{:+.1f}%'
    }))

    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax2 = ax1.twinx()
    x = range(len(snap_val))
    ax1.bar(x, snap_val['total_EUR'], color=C_BLUE, edgecolor='white', alpha=0.8)
    ax2.plot(x, snap_val['delta_%'].fillna(0), color=C_RED, marker='o', linewidth=2)
    ax2.axhline(0, color='gray', linewidth=0.7, linestyle='--')
    ax1.set_xticks(x)
    ax1.set_xticklabels(snap_val.index, rotation=25, ha='right', fontsize=8)
    ax1.set_ylabel('Total EUR')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e6:.0f}M'))
    ax2.set_ylabel('Snapshot Δ%', color=C_RED)
    ax2.set_ylim(-50, 50)
    ax1.set_title(f'EUR value across snapshots [{pv}]')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'snapshot_value_trend.png', dpi=120)
    plt.show()

---
## Section 3 — BOM Tree Structure
Each row is a material node. BOM depth tells us how many cross-border hops a component makes before becoming a finished product — the deeper, the higher the cumulative tariff exposure.

In [ ]:
# ── Cell 8: BOM Level Distribution ───────────────────────────────────────────
print('═' * 60)
print('BOM TREE STRUCTURE')
print('═' * 60)

if BOM_LEVEL_COL and BOM_LEVEL_COL in df.columns:
    level_counts = df[BOM_LEVEL_COL].value_counts().sort_index()
    print(f'\n[BOM Level — {BOM_LEVEL_COL}]')
    print(f'  Max depth : {df[BOM_LEVEL_COL].max()}')
    print(f'  Level distribution:')
    for lvl, cnt in level_counts.items():
        bar = '█' * int(cnt / level_counts.max() * 30)
        print(f'    Level {str(lvl):<4} {cnt:>8,}  {bar}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    level_counts.plot(kind='bar', ax=axes[0], color=C_BLUE, edgecolor='white')
    axes[0].set_title('Row count by BOM level')
    axes[0].set_xlabel('BOM Level')
    axes[0].set_ylabel('Rows')
    axes[0].tick_params(axis='x', rotation=0)

    axes[1].pie(level_counts.values,
                labels=[f'L{l}' for l in level_counts.index],
                autopct='%1.0f%%', startangle=90,
                colors=plt.cm.Blues(np.linspace(0.3, 0.9, len(level_counts))))
    axes[1].set_title('BOM level share')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'bom_level_dist.png', dpi=120)
    plt.show()
else:
    print('⚠  Set BOM_LEVEL_COL in Cell 1.')

if BOM_MATERIAL_COL and BOM_MATERIAL_COL in df.columns:
    print(f'\n[Material ID — {BOM_MATERIAL_COL}]')
    print(f'  Unique materials : {df[BOM_MATERIAL_COL].nunique():,}')
    print(f'  Null IDs         : {df[BOM_MATERIAL_COL].isna().sum():,}')

if BOM_PARENT_COL and BOM_PARENT_COL in df.columns:
    print(f'\n[Parent ID — {BOM_PARENT_COL}]')
    print(f'  Unique parents     : {df[BOM_PARENT_COL].nunique():,}')
    print(f'  Null (root nodes)  : {df[BOM_PARENT_COL].isna().sum():,}')
    if BOM_MATERIAL_COL and BOM_MATERIAL_COL in df.columns:
        cpp = df.groupby(BOM_PARENT_COL)[BOM_MATERIAL_COL].count()
        print(f'  Avg children/parent: {cpp.mean():.1f}  |  Max: {cpp.max()}')

In [ ]:
# ── Cell 9: EUR Value by BOM Level ────────────────────────────────────────────
# Deeper levels = intermediate/raw materials crossing borders
# High value at L2/L3+ = significant tariff pass-through risk

if BOM_LEVEL_COL and BOM_LEVEL_COL in df.columns and VALUE_COLS:
    pv = VALUE_COLS[0]
    vbl = (
        df.assign(_val=pd.to_numeric(df[pv], errors='coerce'))
          .groupby(BOM_LEVEL_COL)['_val']
          .agg(total_EUR='sum', avg_EUR='mean', n_rows='count')
    )
    vbl['total_EUR_pct'] = (100 * vbl['total_EUR'] / vbl['total_EUR'].sum()).round(1)
    print(f'EUR [{pv}] by BOM level:')
    display(vbl.style.format({'total_EUR':'{:,.0f}','avg_EUR':'{:,.0f}','total_EUR_pct':'{:.1f}%'}))

    fig, ax = plt.subplots(figsize=(8, 4))
    vbl['total_EUR'].plot(kind='bar', ax=ax, color=C_BLUE, edgecolor='white')
    ax.set_title('Total EUR by BOM level')
    ax.set_xlabel('BOM Level')
    ax.set_ylabel('Total EUR')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'bom_value_by_level.png', dpi=120)
    plt.show()
else:
    print('Set BOM_LEVEL_COL and VALUE_COLS in Cell 1.')

---
## Section 4 — Shipping Lanes & Corridor Coverage

In [ ]:
# ── Cell 10: Country Coverage & Stage 3 Match ────────────────────────────────
if ORIGIN_COL in df.columns and DEST_COL in df.columns:
    orig_iso3 = iso2_to_iso3(df[ORIGIN_COL])
    dest_iso3 = iso2_to_iso3(df[DEST_COL])

    all_countries = sorted(set(orig_iso3.dropna()) | set(dest_iso3.dropna()))
    print(f'Unique countries ({len(all_countries)}): {all_countries}')

    corridor = pd.Series(
        ['_'.join(sorted([o, d])) for o, d in zip(orig_iso3.fillna('UNK'), dest_iso3.fillna('UNK'))]
    )
    df['corridor'] = corridor

    nvs_corridors = set(corridor.unique())
    matched = nvs_corridors & STAGE3_KEYS
    missing = STAGE3_KEYS - nvs_corridors

    print(f'\nStage 3 corridor coverage:')
    print(f'  ✓ Matched ({len(matched)}): {sorted(matched)}')
    print(f'  ✗ Missing ({len(missing)}): {sorted(missing)}')
    print(f'  + Extra corridors not in Stage 3: {len(nvs_corridors - STAGE3_KEYS)}')

    top_corridors = corridor.value_counts().head(25)
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = [C_GREEN if k in STAGE3_KEYS else C_BLUE for k in top_corridors.index]
    top_corridors[::-1].plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
    ax.set_title('Top 25 corridors by row count  (green = Stage 3 corridor)')
    ax.set_xlabel('Row count')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'top_corridors.png', dpi=120)
    plt.show()
else:
    print('⚠  Set ORIGIN_COL and DEST_COL in Cell 1.')

In [ ]:
# ── Cell 11: EUR Value by Corridor ───────────────────────────────────────────
if 'corridor' in df.columns and VALUE_COLS:
    pv = VALUE_COLS[0]
    cv = (
        df.assign(_val=pd.to_numeric(df[pv], errors='coerce'))
          .groupby('corridor')['_val']
          .agg(total_EUR='sum', avg_EUR='mean', n_rows='count')
          .sort_values('total_EUR', ascending=False)
    )
    cv['in_stage3']    = cv.index.isin(STAGE3_KEYS)
    cv['EUR_pct']      = (100 * cv['total_EUR'] / cv['total_EUR'].sum()).round(2)
    cv['EUR_cum_pct']  = cv['EUR_pct'].cumsum().round(1)

    print('Top 20 corridors by total EUR:')
    display(cv.head(20).style.format({
        'total_EUR':'{:,.0f}','avg_EUR':'{:,.0f}',
        'EUR_pct':'{:.2f}%','EUR_cum_pct':'{:.1f}%'
    }))

    stage3_val = cv.loc[cv['in_stage3'], 'total_EUR'].sum()
    total_val  = cv['total_EUR'].sum()
    print(f'\nStage 3 corridors = {stage3_val/1e6:.1f}M EUR  ({100*stage3_val/total_val:.1f}% of total)')

    fig, ax = plt.subplots(figsize=(10, 7))
    top20 = cv.head(20)
    colors = [C_GREEN if v else C_BLUE for v in top20['in_stage3']]
    top20['total_EUR'][::-1].plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
    ax.set_title(f'Top 20 corridors — total EUR  (green = Stage 3)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'corridor_value.png', dpi=120)
    plt.show()

---
## Section 5 — Industry & Business Function

In [ ]:
# ── Cell 12: Industry Column Distributions ────────────────────────────────────
ind_cols_map = {
    'Industry 2-digit code-text': IND_CODE_TEXT_COL,
    'Industry Group (number)':    IND_GROUP_NUM_COL,
    'Industry Group (text)':      IND_GROUP_TEXT_COL,
    'Business Function':          BF_COL,
}
valid_ind = {lbl: col for lbl, col in ind_cols_map.items() if col and col in df.columns}
n = len(valid_ind)

if n == 0:
    print('⚠  No industry columns configured in Cell 1.')
else:
    ncols = min(2, n)
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(9*ncols, 4.5*nrows), squeeze=False)
    axes_flat = axes.flatten()

    for i, (lbl, col) in enumerate(valid_ind.items()):
        ax = axes_flat[i]
        vc = df[col].value_counts(dropna=False).head(20)
        vc[::-1].plot(kind='barh', ax=ax, color=C_BLUE, edgecolor='white')
        ax.set_title(f'{lbl}\n[{col}]  (top 20 of {df[col].nunique()} unique)', fontsize=9)
        ax.set_xlabel('Row count')
        ax.tick_params(labelsize=8)
        print(f'\n[{lbl}]  unique={df[col].nunique()}  null={df[col].isna().sum():,}')
        display(df[col].value_counts(dropna=False).to_frame(name='count'))

    for j in range(i+1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.suptitle('Industry & Business Function distributions', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'industry_distributions.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Cell 13: EUR Value by Industry Group ──────────────────────────────────────
if IND_GROUP_TEXT_COL and IND_GROUP_TEXT_COL in df.columns and VALUE_COLS:
    pv = VALUE_COLS[0]
    iv = (
        df.assign(_val=pd.to_numeric(df[pv], errors='coerce'))
          .groupby(IND_GROUP_TEXT_COL)['_val']
          .agg(total_EUR='sum', avg_EUR='mean', n_rows='count')
          .sort_values('total_EUR', ascending=False)
    )
    iv['EUR_pct'] = (100 * iv['total_EUR'] / iv['total_EUR'].sum()).round(1)
    print('EUR by Industry Group:')
    display(iv.style.format({'total_EUR':'{:,.0f}','avg_EUR':'{:,.0f}','EUR_pct':'{:.1f}%'}))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    iv['total_EUR'][::-1].plot(kind='barh', ax=axes[0], color=C_BLUE, edgecolor='white')
    axes[0].set_title('Total EUR by Industry Group')
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

    axes[1].pie(iv['total_EUR'], labels=iv.index, autopct='%1.1f%%', startangle=90,
                colors=plt.cm.Set2(np.linspace(0, 1, len(iv))))
    axes[1].set_title('EUR share by Industry Group')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'industry_value.png', dpi=120)
    plt.show()
else:
    print('Set IND_GROUP_TEXT_COL and VALUE_COLS in Cell 1.')

In [ ]:
# ── Cell 14: Industry × Corridor EUR Heatmap ──────────────────────────────────
if IND_GROUP_TEXT_COL and IND_GROUP_TEXT_COL in df.columns and 'corridor' in df.columns and VALUE_COLS:
    pv = VALUE_COLS[0]
    top15 = df['corridor'].value_counts().head(15).index.tolist()
    sub = df[df['corridor'].isin(top15)].copy()
    sub['_val'] = pd.to_numeric(sub[pv], errors='coerce')
    pivot = sub.pivot_table(values='_val', index=IND_GROUP_TEXT_COL,
                            columns='corridor', aggfunc='sum', fill_value=0)

    fig, ax = plt.subplots(figsize=(14, max(4, len(pivot)*0.55)))
    im = ax.imshow(np.log1p(pivot.values), aspect='auto', cmap='Blues')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_yticks(range(len(pivot.index)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_title('log(EUR value) — Industry Group × Top 15 Corridors', fontsize=10)
    plt.colorbar(im, ax=ax, label='log(1 + EUR)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'industry_corridor_heatmap.png', dpi=120)
    plt.show()

---
## Section 6 — Trade Value (EUR)

In [ ]:
# ── Cell 15: Value Stats & Negative Checker ───────────────────────────────────
if not VALUE_COLS:
    print('⚠  Set VALUE_COLS in Cell 1.')
else:
    stats = []
    for col in VALUE_COLS:
        if col not in df.columns:
            print(f'⚠  [{col}] not in dataframe')
            continue
        s = pd.to_numeric(df[col], errors='coerce')
        n_neg = (s < 0).sum()
        if n_neg > 0:
            print(f'⚠  [{col}] has {n_neg:,} NEGATIVE values!')
            display(df[s < 0][[col]].describe())
        stats.append({
            'column':     col,
            'non_null':   s.notna().sum(),
            'null_%':     round(100*s.isna().sum()/len(s), 1),
            'zero_%':     round(100*(s==0).sum()/len(s), 1),
            'negatives':  n_neg,
            'total_EUR':  s.sum(),
            'mean_EUR':   s.mean(),
            'median_EUR': s.median(),
            'p95_EUR':    s.quantile(0.95),
            'p99_EUR':    s.quantile(0.99),
            'max_EUR':    s.max(),
        })

    display(pd.DataFrame(stats).set_index('column').style.format({
        'total_EUR':'{:,.0f}','mean_EUR':'{:,.0f}','median_EUR':'{:,.0f}',
        'p95_EUR':'{:,.0f}','p99_EUR':'{:,.0f}','max_EUR':'{:,.0f}'
    }))

In [ ]:
# ── Cell 16: EUR Distribution (raw + log) ─────────────────────────────────────
valid_vcols = [c for c in VALUE_COLS if c in df.columns]
if valid_vcols:
    n = len(valid_vcols)
    fig, axes = plt.subplots(2, n, figsize=(6*n, 8), squeeze=False)

    for j, col in enumerate(valid_vcols):
        s = pd.to_numeric(df[col], errors='coerce').dropna()
        s_pos = s[s > 0]
        cap = s_pos.quantile(0.99) if len(s_pos) else 1

        s_pos.clip(upper=cap).hist(bins=60, ax=axes[0,j], color=C_BLUE, edgecolor='white', alpha=0.85)
        axes[0,j].set_title(f'{col}\n(raw, capped at p99={cap:,.0f} EUR)', fontsize=9)
        axes[0,j].set_xlabel('EUR')
        axes[0,j].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

        np.log1p(s_pos).hist(bins=60, ax=axes[1,j], color=C_AMBER, edgecolor='white', alpha=0.85)
        axes[1,j].set_title(f'{col} — log(1+EUR)', fontsize=9)
        axes[1,j].set_xlabel('log(1 + EUR)')

    fig.suptitle('EUR value distributions (positive values only)', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'value_distributions.png', dpi=120)
    plt.show()

---
## Section 7 — All Column Distributions (90 columns)
**Scroll down** — numeric columns are batched 6 per figure, categoricals 4 per figure.

In [ ]:
# ── Cell 17: All Numeric Distributions ───────────────────────────────────────
numeric_cols = df.select_dtypes(include='number').columns.tolist()
skip = {c for c in [BOM_LEVEL_COL, IND_GROUP_NUM_COL] if c}
numeric_cols = [c for c in numeric_cols if c not in skip]

print(f'Plotting {len(numeric_cols)} numeric columns in batches of 6 — scroll down...')
BATCH = 6

for start in range(0, len(numeric_cols), BATCH):
    batch = numeric_cols[start:start+BATCH]
    n = len(batch)
    fig, axes = plt.subplots(1, n, figsize=(4.5*n, 3.5), squeeze=False)
    axes = axes[0]

    for ax, col in zip(axes, batch):
        s = pd.to_numeric(df[col], errors='coerce').dropna()
        s_pos = s[s > 0]
        if len(s_pos) > 1:
            np.log1p(s_pos).hist(bins=40, ax=ax, color=C_BLUE, edgecolor='white', alpha=0.85)
        elif len(s) > 0:
            s.hist(bins=20, ax=ax, color=C_AMBER, edgecolor='white')
        ax.set_title(col, fontsize=8)
        ax.set_xlabel('log(1+val)', fontsize=7)
        ax.tick_params(labelsize=7)
        null_pct = df[col].isna().mean()*100
        ax.annotate(f'{null_pct:.0f}% null', xy=(0.97,0.97), xycoords='axes fraction',
                    ha='right', va='top', fontsize=7,
                    color=C_RED if null_pct > 10 else 'gray')

    for j in range(n, BATCH):
        axes[j].set_visible(False)

    fig.suptitle(f'Numeric [{start+1}–{start+n}] of {len(numeric_cols)}', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 18: All Categorical Distributions ────────────────────────────────────
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
already = {TIMESTAMP_COL} | {ORIGIN_COL, DEST_COL, 'corridor',
                                   IND_CODE_TEXT_COL, IND_GROUP_TEXT_COL,
                                   BF_COL, PRODUCT_DESC_COL} - {None}
cat_cols = [c for c in cat_cols if c not in already]

print(f'Plotting {len(cat_cols)} categorical columns in batches of 4 — scroll down...')
BATCH = 4

for start in range(0, len(cat_cols), BATCH):
    batch = cat_cols[start:start+BATCH]
    n = len(batch)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), squeeze=False)
    axes = axes[0]

    for ax, col in zip(axes, batch):
        vc = df[col].value_counts(dropna=False).head(15)
        vc[::-1].plot(kind='barh', ax=ax, color=C_BLUE, edgecolor='white')
        ax.set_title(col, fontsize=8)
        ax.tick_params(labelsize=7)
        null_pct = df[col].isna().mean()*100
        ax.annotate(f'{df[col].nunique()} uniq  |  {null_pct:.0f}% null',
                    xy=(0.97,0.02), xycoords='axes fraction',
                    ha='right', va='bottom', fontsize=7, color='gray')

    for j in range(n, BATCH):
        axes[j].set_visible(False)

    fig.suptitle(f'Categorical [{start+1}–{start+n}] of {len(cat_cols)} (top 15 values)', fontsize=10)
    plt.tight_layout()
    plt.show()

---
## Section 8 — Missing Data

In [ ]:
# ── Cell 19: Missing Data ─────────────────────────────────────────────────────
missing_pct = df.isnull().mean().sort_values(ascending=False)
cols_missing = missing_pct[missing_pct > 0]

if cols_missing.empty:
    print('✓ No missing values.')
else:
    print(f'{len(cols_missing)} columns have missing values:')
    display(cols_missing.apply(lambda x: f'{100*x:.1f}%').to_frame(name='missing_%'))

    fig, ax = plt.subplots(figsize=(10, max(4, len(cols_missing)*0.35)))
    colors = [C_RED if v > 0.5 else C_AMBER if v > 0.1 else C_BLUE for v in cols_missing.values]
    cols_missing[::-1].plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
    ax.axvline(0.1, color='gray', linestyle='--', linewidth=0.8, label='10%')
    ax.axvline(0.5, color=C_RED,  linestyle='--', linewidth=0.8, label='50%')
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_title('Missing data by column  (red >50%, orange >10%)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'missing_data.png', dpi=120)
    plt.show()

---
## Section 9 — Tariff Exposure Proxy
Value by corridor × BOM level — the foundation for the Stage 4 impact score. Deep BOM levels at high-value Stage 3 corridors = highest tariff pass-through risk.

In [ ]:
# ── Cell 20: Value by Corridor × BOM Level ───────────────────────────────────
if 'corridor' in df.columns and BOM_LEVEL_COL and BOM_LEVEL_COL in df.columns and VALUE_COLS:
    pv = VALUE_COLS[0]
    pivot = (
        df.assign(_val=pd.to_numeric(df[pv], errors='coerce'))
          .groupby(['corridor', BOM_LEVEL_COL])['_val']
          .sum().unstack(fill_value=0)
    )
    pivot = pivot.loc[pivot.sum(axis=1).nlargest(20).index]

    fig, ax = plt.subplots(figsize=(12, 7))
    pivot.plot(kind='barh', stacked=True, ax=ax, colormap='Blues', edgecolor='white')
    ax.set_title('EUR value by corridor × BOM level (top 20 corridors)')
    ax.set_xlabel('Total EUR')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
    ax.legend(title='BOM Level', bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'corridor_bom_value.png', dpi=120, bbox_inches='tight')
    plt.show()

    s3 = pivot[pivot.index.isin(STAGE3_KEYS)]
    if not s3.empty:
        print('Stage 3 corridors — EUR by BOM level:')
        display(s3.style.format('{:,.0f}'))
else:
    print('Run Sections 3 & 4 first, and set VALUE_COLS in Cell 1.')

In [ ]:
# ── Cell 21: Pareto — Corridors Driving 80% of Value ─────────────────────────
if 'corridor' in df.columns and VALUE_COLS:
    pv = VALUE_COLS[0]
    cv_s = (
        df.assign(_val=pd.to_numeric(df[pv], errors='coerce'))
          .groupby('corridor')['_val'].sum()
          .sort_values(ascending=False)
    )
    cum_pct = cv_s.cumsum() / cv_s.sum() * 100
    n80 = int((cum_pct <= 80).sum()) + 1

    print(f'{n80} corridors account for ≥80% of total EUR value')
    top_n80 = cv_s.head(n80).index
    s3_in_top = sum(1 for c in top_n80 if c in STAGE3_KEYS)
    print(f'{s3_in_top} of the top {n80} corridors are Stage 3 corridors')

    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax2 = ax1.twinx()
    x = range(len(cv_s))
    colors = [C_GREEN if c in STAGE3_KEYS else C_BLUE for c in cv_s.index]
    ax1.bar(x, cv_s.values, color=colors, edgecolor='none', width=1.0)
    ax2.plot(x, cum_pct.values, color=C_RED, linewidth=2)
    ax2.axhline(80, color=C_RED, linestyle='--', linewidth=0.8)
    ax2.axvline(n80, color='gray', linestyle='--', linewidth=0.8)
    ax1.set_xlabel('Corridor (ranked by EUR value)')
    ax1.set_ylabel('EUR value')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
    ax2.set_ylabel('Cumulative %', color=C_RED)
    ax2.set_ylim(0, 105)
    ax1.set_title('Pareto: EUR value concentration by corridor  (green = Stage 3)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'corridor_pareto.png', dpi=120)
    plt.show()

---
## Section 10 — Stage 4 Join Readiness

In [ ]:
# ── Cell 22: Join Readiness Checklist ────────────────────────────────────────
print('Join key for features_master.csv: (corridor, snapshot_week)')
print('No HS codes in NVS → crosswalk via industry group')
print()

checks = [
    ('Origin column',          ORIGIN_COL,                              'corridor mapping'),
    ('Destination column',     DEST_COL,                                'corridor mapping'),
    ('Timestamp column',        TIMESTAMP_COL,                                  'snapshot weeks'),
    ('BOM level col',          BOM_LEVEL_COL,                           'depth weighting'),
    ('Material ID col',        BOM_MATERIAL_COL,                        'BOM node ID'),
    ('Industry Group text',    IND_GROUP_TEXT_COL,                      'HS crosswalk'),
    ('Industry 2-digit text',  IND_CODE_TEXT_COL,                       'HS crosswalk'),
    ('Primary EUR value col',  VALUE_COLS[0] if VALUE_COLS else None,   'impact quantification'),
]

all_ok = True
for name, col, note in checks:
    if col and col in df.columns:
        status = '✓'
    elif col:
        status = '✗  NOT IN DATAFRAME'
        all_ok = False
    else:
        status = '⚠  NOT CONFIGURED'
        all_ok = False
    print(f'  {status}  {name:<28} → {str(col):<32} ({note})')

print()
print('✓ Ready for Stage 4 join.' if all_ok else '✗ Resolve flagged items in Cell 1 before Stage 4.')

print()
print('── HS crosswalk (Industry Group → HS chapters) ─────────────────────────')
crosswalk = {
    'Dairy':                       ['3507','2102','3002'],
    'Animal':                      ['2309','3507'],
    'Food & Beverages':            ['3507','2102','2106','3002'],
    'Dietary Supplements':         ['3002','3504','2936'],
    'Advanced Health & Nutrition': ['3002','3504','2936','2106'],
    'Specialized Nutrition':       ['3504','2936','3002'],
    'Bioenergy':                   ['3507','3821'],
    'Household Care':              ['3507','3824'],
    'Tech':                        ['3822','3824'],
    'Plant':                       ['3002','3507','3821'],
}
for ig, hs in crosswalk.items():
    print(f'  {ig:<35} → HS {", ".join(hs)}')

---
## EDA Summary

| # | Check | What to note |
|---|-------|--------------|
| 1 | Corridor coverage vs Stage 3 | Missing corridors = gaps in Stage 4 join |
| 2 | BOM level value concentration | High EUR at L2+ = high tariff pass-through risk |
| 3 | Industry Group EUR share | Top 2–3 groups dominate → set model weights accordingly |
| 4 | Pareto corridor count | How many corridors need tariff modelling |
| 5 | Timestamp gap matrix | Identify which timestamp maps to `snapshot_week` |
| 6 | Missing data | Columns >50% null → drop before Stage 4 join |
| 7 | Negative values | Any EUR negatives need business explanation before modelling |

**Plots saved to:** `outputs/nvs_eda/`